In [ ]:
import os
for d, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(d, f))

In [ ]:
import pandas as pd, sqlite3

df = pd.read_csv("/kaggle/input/notebooks/diraamelia/01-online-retail-ii-data-cleaning/sales_products_clean.csv",
                 parse_dates=["invoicedate"])
conn = sqlite3.connect("/kaggle/working/retail.db")
df.to_sql("sales", conn, if_exists="replace", index=False)

pd.read_sql("SELECT COUNT(*) AS n, COUNT(DISTINCT customer_id) AS pelanggan FROM sales", conn)

## 2. Pendapatan bulanan dan pertumbuhan
Pendapatan memuncak tiap November (sekitar 1,15 juta) dan mulai naik sejak
September, pola musiman kuartal 4 yang berulang di 2010 dan 2011.
Desember 2011 hanya berisi data sampai sekitar tanggal 9, sehingga
penurunannya (-69,8%) bukan penurunan bisnis yang sebenarnya.

In [ ]:
# 1. Pendapatan, jumlah pesanan, dan pelanggan per bulan
q1 = """
SELECT invoice_month AS bulan,
       ROUND(SUM(total_price), 0) AS pendapatan,
       COUNT(DISTINCT invoice) AS jumlah_pesanan,
       COUNT(DISTINCT customer_id) AS pelanggan_aktif
FROM sales
GROUP BY invoice_month
ORDER BY invoice_month;
"""
display(pd.read_sql(q1, conn))

# 2. Pertumbuhan bulan ke bulan (window function LAG)
q2 = """
WITH bulanan AS (
  SELECT invoice_month AS bulan, SUM(total_price) AS pendapatan
  FROM sales GROUP BY invoice_month
)
SELECT bulan,
       ROUND(pendapatan, 0) AS pendapatan,
       ROUND((pendapatan - LAG(pendapatan) OVER (ORDER BY bulan)) * 100.0
             / LAG(pendapatan) OVER (ORDER BY bulan), 1) AS growth_pct
FROM bulanan
ORDER BY bulan;
"""
display(pd.read_sql(q2, conn))

## 3. Produk terlaris
Produk terlaris didominasi dekorasi rumah dan barang musiman. Regency Cakestand
memimpin dari sisi pendapatan, sedangkan White Hanging Heart memimpin dari sisi
unit terjual. Peringkat berbeda tergantung metriknya.

In [ ]:
# Top 10 produk berdasarkan pendapatan
q3 = """
SELECT stockcode, MAX(description) AS description,
       SUM(quantity) AS unit_terjual,
       ROUND(SUM(total_price), 0) AS pendapatan
FROM sales
GROUP BY stockcode
ORDER BY pendapatan DESC
LIMIT 10;
"""
display(pd.read_sql(q3, conn))

## 4. Pendapatan per negara
United Kingdom menyumbang 83,4% pendapatan. Irlandia (EIRE) dan Belanda bernilai
besar dari sedikit pelanggan, sehingga diduga pembeli grosir

In [ ]:
# Pendapatan per negara (dengan persentase)
q4 = """
SELECT country,
       ROUND(SUM(total_price), 0) AS pendapatan,
       COUNT(DISTINCT customer_id) AS pelanggan,
       ROUND(SUM(total_price) * 100.0 / (SELECT SUM(total_price) FROM sales), 1) AS persen
FROM sales
GROUP BY country
ORDER BY pendapatan DESC
LIMIT 10;
"""
display(pd.read_sql(q4, conn))

## 5. Pelanggan teratas
Sepuluh pelanggan teratas menyumbang sekitar 16% pendapatan. Frekuensi dan nilai
belanja tidak selalu sejalan, yang mendasari penggunaan RFM pada tahap berikutnya.

In [ ]:
# 10 pelanggan terbaik dengan peringkat dan kontribusi
q5 = """
SELECT customer_id,
       ROUND(SUM(total_price), 0) AS total_belanja,
       COUNT(DISTINCT invoice) AS jumlah_pesanan,
       RANK() OVER (ORDER BY SUM(total_price) DESC) AS peringkat,
       ROUND(SUM(total_price) * 100.0 / (SELECT SUM(total_price) FROM sales), 2) AS persen_total
FROM sales
GROUP BY customer_id
ORDER BY peringkat
LIMIT 10;
"""
display(pd.read_sql(q5, conn))